# Ensemble Learning

Mounting Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---

## Cloning github repository

### Only choose one of the following cells to run

1. This cell will clone the main repository of the project.

In [ ]:
!git clone https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

2. This will clone one single branch ONLY!

In [ ]:
!git clone -b feat/data-augm --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

---

## Setup Stage

### Copying dataset from the Google Drive to the current local disk of VM.

In [ ]:
# 1. Copy zip from Drive to local VM
!cp /content/drive/MyDrive/ImageNetSubset.zip /content/

# 2. Unzip directly to /content/
!unzip -q /content/ImageNetSubset.zip -d /content/

# 3. Rename "ImageNetSubset" to "datasets" 
!mv /content/ImageNetSubset /content/xAI-proj-m-ws2526/datasets

# 4. (Optional) Remove the zip to save space
!rm /content/ImageNetSubset.zip

### Setup the root directory for the project.

In [ ]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent  # keep going to prefer the outermost match
    return root or start

# Dynamically get the name of the cloned repository if it exists
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# If the cloned repository exists as a subdirectory, change into it
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Now, find the project root from within the repository (or its parent if already there)
ROOT = find_project_root(Path.cwd()).resolve()

# Ensure we are in the identified project root
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

### Installing dependencies

In [ ]:
# Install dependencies
!pip install -r experiments/base_ensemble/requirements.txt --quiet

### Wandb setup

In [ ]:
import wandb

# Login to WandB - this will prompt you to enter your API key
wandb.login()

---

## Training Stage

In [ ]:
!python -m experiments.base_ensemble.scripts.train_single_model \
--config experiments/base_ensemble/configs/default.yaml \
--amp \
--model resnet34 # TODO: change to your model

### [OPTIONAL] Wandb Sweep

This cell will produce an agent specifically for the wandb sweep configured in `sweep_learning_rate.yaml`.

In [ ]:
!wandb sweep experiments/base_ensemble/configs/sweep_learning_rate.yaml

The next cell has to be edited for the wandb sweep.

In [ ]:
!wandb agent #TODO: add the agent id here

---

## Validation Stage

### Choose one of the following cells to run.

1. This cell will evaluate the ensemble architecture on the test set.

In [ ]:
# Run ensemble evaluation on a dataset
!python -m experiments.base_ensemble.scripts.ensemble_inference --evaluate --data-dir datasets

2. This cell will evaluate the ensemble architecture on one single image.

In [ ]:
# Upload a test image or use sample
!python -m experiments.base_ensemble.scripts.ensemble_inference --image datasets/test_image.jpg --show-individual

---

## Script for saving best models in Google Drive

In [ ]:
# Define source and destination paths
# We check experiments/base_ensemble/checkpoints (relative to project root) first
source_dir = Path("experiments/checkpoints")

# Check for fallback paths if the default doesn't exist (e.g., if strictly using /chechpoints)
if not source_dir.exists():
    if Path("/chechpoints").exists(): # Handling the specific path mentioned
        source_dir = Path("/chechpoints")
    elif Path("/checkpoints").exists(): # Handling potential typo correction
        source_dir = Path("/checkpoints")

# Destination folder on the mounted drive
# You can change "saved_checkpoints" to your preferred folder name
dest_dir = Path("/content/drive/MyDrive/saved_checkpoints")

print(f"Source Directory: {source_dir}")
print(f"Destination Directory: {dest_dir}")

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy .pth files
if source_dir.exists():
    pth_files = list(source_dir.glob("*.pth"))
    
    if not pth_files:
        print("No .pth files found in source directory.")
    else:
        print(f"Found {len(pth_files)} .pth files to copy.")
        
        for file_path in pth_files:
            try:
                shutil.copy2(file_path, dest_dir / file_path.name)
                print(f"Successfully copied: {file_path.name}")
            except Exception as e:
                print(f"Error copying {file_path.name}: {e}")
else:
    print(f"Source directory {source_dir} not found. Please check the path.")

---

## This is only for the PC-POOL at GU.

In [ ]:
# Initialize conda for PowerShell
# Do this in the TERMINAL
& "C:\ProgramData\anaconda3\Scripts\conda.exe" init powershell

In [ ]:
!conda create -n xai-proj python=3.11 -y ipykernel
!conda activate xai-proj
!conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia
!pip install wandb pyyaml

In [ ]:
Use this path on Windows: C:\Users\ba081274\Downloads\ImageNetSubset\ImageNetSubset\